# 22 · Cases by period against pandas

**Question.** Does Fieldwork's `case_series` widget bin dated records into periods the way
pandas does?

Snow 1855 Table 1 — deaths by date, 43 daily rows — binned by day, week and month. This is the
epidemic curve the John Snow example renders.

**Genuinely independent.** Fieldwork bins with its own calendar arithmetic; pandas bins with
`resample`, a different implementation of a different design. Neither shares code with the
other — unlike check 21 next door, where both sides call the same C library.

Criteria fixed before running: per period, every bin key in order exactly, every weighted count
exactly, the total exactly, every period between the first and last present including empty
ones, and the dated and undated record counts exact.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_22_case_series import run

report = run(Path.cwd() / "fixtures" / "case-series-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], f"over {report['rows']} rows "
      f"({report['datedRows']} dated, {report['undatedRows']} undated)")
print()
for period in report["periods"]:
    print(f"by {period['period']:<6} resample({period['resampleRule']})")
    print(f"  {period['binsReference']:>3} bins, {period['emptyBins']} with no deaths, "
          f"total {period['totalReference']:.0f}")
    print(f"  {period['firstKey']} to {period['lastKey']}, peak {period['peak'][0]} "
          f"at {period['peak'][1]:.0f}")
    print()

## The week was the dangerous case, and it caught me

The docstring of this check names the week as the rule most likely to be wrong. **The first
version of it was wrong anyway.**

Fieldwork keys a week by the Monday of its ISO week. The check used
`resample("W-SUN", label="left", closed="left")`, whose left label is the preceding **Sunday**.
Every week key came out one day early — and every count stayed entirely plausible: 7 bins, a
total of 616, the peak in the right week. Nothing about the shape of the curve looked wrong.

Only comparing the **keys** caught it. Predicting a hazard did not prevent walking into it.

In [ ]:
import pandas as pd

fixture = json.loads((Path.cwd() / "fixtures" / "case-series-soho-001.json").read_text())
rows = fixture["input"]["rows"]
frame = pd.DataFrame({"date": pd.to_datetime([r["date"] for r in rows]),
                      "deaths": [r["deaths"] for r in rows]}).set_index("date")

fieldwork = [(b["key"], b["count"]) for b in fixture["output"]["week"]["bins"]]
print(f"{'rule':<34} {'first three bins'}")
for rule, options in [("W-MON, label=left, closed=left", dict(label="left", closed="left")),
                      ("W-SUN, label=left, closed=left", dict(label="left", closed="left")),
                      ("W-MON, default labelling", {})]:
    frequency = rule.split(",")[0]
    binned = frame["deaths"].resample(frequency, **options).sum()
    first = [(t.strftime("%Y-%m-%d"), int(v)) for t, v in binned.items()][:3]
    print(f"{rule:<34} {first}")
print(f"{'fieldwork':<34} {fieldwork[:3]}")
print()
print("Three plausible mappings, three different answers, one of them right. The counts alone")
print("would not have told them apart: it is the keys that differ.")

## Dive deeper · what the period choice does to the curve

The same 616 deaths, read three ways. The period is not a display preference — it changes what
the curve appears to say about when the outbreak happened.

In [ ]:
for period in report["periods"]:
    bins = [(b["key"], b["count"]) for b in fixture["output"][period["period"]]["bins"]]
    peak_key, peak_count = max(bins, key=lambda row: row[1])
    share = peak_count / period["totalReference"]
    print(f"by {period['period']:<6} {len(bins):>3} bins, peak {peak_key} holding "
          f"{peak_count:.0f} deaths ({share:.0%} of the outbreak)")

print()
print("By day the peak is one day holding a fifth of the deaths; by month it is 'September',")
print("holding nearly all of them and saying almost nothing. The daily curve is the one that")
print("shows the outbreak ending as abruptly as it began -- which is the observation the")
print("Broad Street pump handle rests on.")
print()
print("The empty days matter for the same reason: 12 of the 43 days have no deaths, and a curve")
print("that omitted them would compress the timeline and hide the gaps.")

### Reading the result

`agrees` means two independent implementations of calendar binning produced the same keys and
the same weighted counts for all three periods.

It does not mean the dates are right, or that a curve by date of **death** says what a reader
will take it to say. Snow's table records deaths by date of death; the outbreak's shape by date
of onset would be shifted earlier by the course of the illness. The widget carries the declared
kind so that a chart cannot quietly relabel it, and no check can supply onset dates nobody
recorded.